# Recreating HF365 from Raw HLS Satellite Data

HF365's three indices (EVI2, NDVI, NIRv) are each computed from just
**two** raw satellite bands -- Red and Near-Infrared (NIR) -- even
though the underlying satellites capture far more than that (Landsat
8/9 has 7 reflective bands, Sentinel-2 has 11). This notebook:

1. Recreates EVI2/NDVI/NIRv from raw Red+NIR reflectance, using the
   public formulas, so you can validate that a from-scratch computation
   matches HF365's own published values.
2. Documents exactly how to pull that raw Red/NIR data yourself, using
   the same AppEEARS tool/workflow already used throughout this
   project for MODIS.

## The public formulas
- `NDVI  = (NIR - Red) / (NIR + Red)`
- `EVI2  = 2.5 * (NIR - Red) / (NIR + 2.4*Red + 1)`
- `NIRv  = NDVI * NIR`

## Where the raw data comes from
HF365 is built on **HLS** (Harmonized Landsat Sentinel-2) -- the same
combined Landsat 8/9 + Sentinel-2A/B/C product discussed earlier in
this project. Raw Red/NIR reflectance is available through **AppEEARS**
(the same tool used for MODIS), just a different product:

| Product | Satellite | Red band | NIR band |
|---|---|---|---|
| `HLSL30` | Landsat 8/9 | B4 | B5 |
| `HLSS30` | Sentinel-2 | B04 | B8A |

## Required file to run this notebook for real
An AppEEARS **Point Sample** (or Area Sample) request for `HLSL30`
and/or `HLSS30`, with the Red and NIR bands selected, at your site
coordinate and date range -- see the request-building cell at the end
of this notebook for exact setup instructions.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# vvv EDIT THIS to your downloaded HLS point-sample CSV vvv
HLS_CSV = "synthetic_HLS_point_sample.csv"
HF365_DIR = "."
# ^^^ EDIT THIS to your downloaded HLS point-sample CSV vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}


## 1. Load raw HLS Red/NIR and compute the three indices from scratch

Column names below (`HLS_Red`, `HLS_NIR`) match this notebook's
synthetic test file -- **rename these to match your real AppEEARS
export's actual column names** (they typically look like
`HLSL30_020_B04` / `HLSL30_020_B05` or similar; print
`hls_raw.columns` first to check).

In [ ]:
hls_raw = pd.read_csv(HLS_CSV)
hls_raw["Date"] = pd.to_datetime(hls_raw["Date"])
print("Columns found:", list(hls_raw.columns))

RED_COL = "HLS_Red"   # <-- rename to match your real export
NIR_COL = "HLS_NIR"   # <-- rename to match your real export

red = hls_raw[RED_COL]
nir = hls_raw[NIR_COL]

hls_raw["NDVI_recreated"] = (nir - red) / (nir + red)
hls_raw["EVI2_recreated"] = 2.5 * (nir - red) / (nir + 2.4*red + 1)
hls_raw["NIRv_recreated"] = hls_raw["NDVI_recreated"] * nir

print(f"\nRecreated {len(hls_raw)} rows")
hls_raw[["Date","NDVI_recreated","EVI2_recreated","NIRv_recreated"]].head()


## 2. Compare against HF365's own published values

Loads the actual HF365 CSVs (averaged across all plots, matching the
same site-level comparison used throughout this project) and merges by
date to see how closely a from-scratch recreation matches.

In [ ]:
hf365_avg = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    hf365_avg[name] = df.set_index(date_col).mean(axis=1)

hf365_df = pd.DataFrame(hf365_avg).reset_index()
hf365_df = hf365_df.rename(columns={hf365_df.columns[0]: "Date"})  # whatever the index-turned-column is actually called

merged = hls_raw[["Date","NDVI_recreated","EVI2_recreated","NIRv_recreated"]].merge(
    hf365_df, on="Date", how="inner"
)
print(f"Matched dates between recreated HLS and published HF365: {len(merged)}")
merged.head()


## 3. How closely do they agree?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [("NDVI_recreated","NDVI"), ("EVI2_recreated","EVI2"), ("NIRv_recreated","NIRv")]
for ax, (recreated_col, published_col) in zip(axes, pairs):
    ax.scatter(merged[published_col], merged[recreated_col], s=15, alpha=0.6)
    lims = [min(merged[published_col].min(), merged[recreated_col].min()),
            max(merged[published_col].max(), merged[recreated_col].max())]
    ax.plot(lims, lims, "k--", linewidth=1)
    r = merged[published_col].corr(merged[recreated_col])
    diff = (merged[recreated_col] - merged[published_col]).abs()
    ax.set_title(f"{published_col}\nr={r:.4f}, mean|diff|={diff.mean():.4f}")
    ax.set_xlabel("Published (HF365)")
    ax.set_ylabel("Recreated (from raw bands)")
plt.tight_layout()
plt.show()


## 4. What a mismatch would (and wouldn't) mean

- **Near-perfect agreement (r > 0.99, tiny mean difference):** confirms
  HF365's published values are exactly what the public formulas predict
  from the underlying raw bands -- full transparency, no hidden
  processing steps.
- **Small, consistent offset:** could mean a different band-selection
  convention (e.g. Sentinel's broad `B08` NIR vs narrow `B8A`), a
  different cloud/quality filter, or plot-averaging done at a different
  stage. Worth checking which raw bands and QC filter your AppEEARS
  request actually used.
- **Large or inconsistent mismatch:** would be worth real investigation
  -- but is not expected given these are published, standard formulas.


## 5. Extending this to new locations (with roughly-matching timing)

Since HLS is a single, continuously-operating global product built from
the same Landsat 8/9 + Sentinel-2A/B/C constellation everywhere, **any**
location gets a similar revisit cadence (HLS's own documentation states
observations less than 1.4 days apart on average, globally). Exact
calendar dates won't be identical to Harvard Forest's unless the site
is in the same or an adjacent tile, but the underlying sampling
*statistics* (density, seasonal coverage) will be comparable.

**Recommended locations to check first:** the same 4 control-point
coordinates already validated in `03-control-points` (MAFar,
DifferentState, JapanOppositeTZ, AmazonEquatorial) -- since those
already have confirmed real vegetated land cover (not water/rock, a
mistake caught and fixed earlier in this project), requesting genuine
HF365-style HLS indices at those same points gives an apples-to-apples
comparison: HLS-derived indices vs. HLS-derived indices, not HLS vs.
MODIS as the current control-point analysis does.

The cell below builds the exact AppEEARS request file for this.

In [ ]:
import json

control_points = [
    {"name": "MAFar", "lat": 42.6200, "lon": -73.1750},
    {"name": "DifferentState", "lat": 45.8205, "lon": -121.9519},
    {"name": "JapanOppositeTZ", "lat": 36.1333, "lon": 137.4167},
    {"name": "AmazonEquatorial", "lat": -2.8567, "lon": -54.9589},
    {"name": "HarvardForest_Tower", "lat": 42.5378, "lon": -72.1715},  # include the original site as a self-check
]

with open("hls_expansion_points.csv", "w") as f:
    for p in control_points:
        f.write(f"{p['name']}, , {p['lat']}, {p['lon']}\n")

print("Saved hls_expansion_points.csv -- upload this to AppEEARS Point Sample.")
print("\nIn AppEEARS:")
print("  1. Extract -> Point Sample -> upload hls_expansion_points.csv")
print("  2. Search for product 'HLSL30' AND 'HLSS30' (both, for full temporal density)")
print("  3. Select bands: B4/B5 (HLSL30) and B04/B8A (HLSS30), plus Fmask for quality")
print("  4. Date range: match your HF365 comparison period (e.g. 2015-01-01 to 2020-12-31)")
print("  5. Submit, download, then rename columns in Section 1 above to match the real export")
